# 01 - Generate raw data (Emberline Studios)

Synthetic but realistic source-system files for the **Game CX Players** demo: Player 360, payment fraud and toxicity.

**Window:** 2025-01-01 to 2026-12-31. **Simulated `as_of_date`:** 2026-12-31. No code here uses `current_date()`.

**Approach (hybrid):** `dbldatagen` builds the independent player attributes. PySpark builds everything that depends on behaviour or time: latent traits, devices, sessions, transactions, chargebacks, chat and reports. Intermediate results are frozen as Parquet so every downstream table sees the same random draws.

**Outputs**

| Volume path | Format | Delivery |
|---|---|---|
| `raw/account/players/` | CSV | snapshot (PII) |
| `raw/game_svc/games/`, `raw/game_svc/live_ops_events/` | JSON | snapshot |
| `raw/telemetry/devices/` | CSV | snapshot |
| `raw/telemetry/sessions/` | JSON | daily |
| `raw/commerce/transactions/` | JSON | daily |
| `raw/payments/chargebacks/` | CSV | daily, late-arriving 15-75 days |
| `raw/chat/chat_messages/` | JSON | daily |
| `raw/safety/player_reports/` | JSON | daily |
| `ground_truth/players`, `transactions`, `chat_labels` | Parquet | evaluation only, never features |

Small, deliberate data-quality defects are injected into the **raw** files only, so the silver expectations have something to catch.

Re-running the notebook overwrites all outputs. Seeds are fixed, so the counts come out the same on every run.

In [0]:
%pip install -q dbldatagen==0.4.0post1

In [0]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from datetime import date, timedelta

import dbldatagen as dg
from pyspark.sql import Window
from pyspark.sql import functions as F

c = F.col

# ---------------- Configuration ----------------
CATALOG = "mg_aws_serverless_workspace_catalog"
SCHEMA = "game_cx_players"
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/raw"
GT = f"/Volumes/{CATALOG}/{SCHEMA}/ground_truth"
WORK = f"{GT}/_work"       # frozen intermediates, deleted at the end
STAGE = f"{RAW}/_staging"  # Spark write staging; same volume as targets so renames are cheap

START_DATE = "2025-01-01"
END_DATE = "2026-12-31"
AS_OF_DATE = END_DATE      # simulated 'today': all relative logic uses this, never current_date()

N_PLAYERS = 50_000
N_RINGS, RING_SIZE = 75, 8
SEED = 42

spark.conf.set("spark.sql.session.timeZone", "UTC")


def lit_date(s):
    """Return a Spark DATE literal for an ISO date string."""
    return F.to_date(F.lit(s))


def kv_map(d):
    """Build a Spark MAP literal from a Python dict of scalar values."""
    return F.create_map(*[x for k, v in d.items() for x in (F.lit(k), F.lit(v))])


def with_uniforms(df, names, seed_offset):
    """Add one independent U(0,1) column per name, seeded deterministically from SEED + seed_offset."""
    for i, n in enumerate(names):
        df = df.withColumn(n, F.rand(SEED + seed_offset + i))
    return df


def wpick(values, weights, u):
    """Pick one of `values` with probability proportional to `weights`, driven by the uniform column `u`."""
    total, acc, expr = float(sum(weights)), 0.0, None
    for v, w in zip(values, weights):
        acc += w / total
        expr = F.when(u < acc, F.lit(v)) if expr is None else expr.when(u < acc, F.lit(v))
    return expr.otherwise(F.lit(values[-1]))


def upick(values, u):
    """Pick uniformly from a Python list of literals, driven by the uniform column `u`."""
    return F.element_at(F.array(*[F.lit(v) for v in values]), (F.floor(u * len(values)) + 1).cast("int"))


def ts_plus_seconds(ts_col, seconds_col):
    """Add a number of seconds (column) to a timestamp (or date) column."""
    return F.timestamp_seconds(F.unix_timestamp(ts_col.cast("timestamp")) + seconds_col.cast("long"))


def materialize(df, name):
    """Write df as Parquet under WORK and read it back, freezing random draws for downstream steps."""
    path = f"{WORK}/{name}"
    df.write.mode("overwrite").parquet(path)
    return spark.read.parquet(path)


def write_raw(df, source, entity, fmt, date_col=None):
    """Write df to raw/<source>/<entity>/<source>_<entity>_<yyyymmdd>.<fmt>.

    Daily entities (date_col given) get one file per day; snapshots get one file dated AS_OF_DATE.
    Returns the number of files written.
    """
    out_dir, stage = f"{RAW}/{source}/{entity}", f"{STAGE}/{source}_{entity}"
    shutil.rmtree(out_dir, ignore_errors=True)
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(out_dir, exist_ok=True)
    key = F.date_format(c(date_col), "yyyyMMdd") if date_col else F.lit(AS_OF_DATE.replace("-", ""))
    writer = (df.withColumn("_file_date", key)
                .repartition("_file_date")
                .write.mode("overwrite").partitionBy("_file_date"))
    if fmt == "csv":
        writer.option("header", True).csv(stage)
    else:
        writer.json(stage)
    moves = []
    for part_dir in sorted(os.listdir(stage)):
        if not part_dir.startswith("_file_date="):
            continue
        ymd = part_dir.split("=", 1)[1]
        parts = sorted(f for f in os.listdir(f"{stage}/{part_dir}") if f.startswith("part-"))
        for i, f in enumerate(parts):
            suffix = "" if i == 0 else f"_{i:02d}"
            moves.append((f"{stage}/{part_dir}/{f}", f"{out_dir}/{source}_{entity}_{ymd}{suffix}.{fmt}"))
    with ThreadPoolExecutor(max_workers=32) as pool:
        list(pool.map(lambda m: shutil.move(*m), moves))
    shutil.rmtree(stage, ignore_errors=True)
    return len(moves)


print(f"raw={RAW}\nground_truth={GT}\nwindow={START_DATE}..{END_DATE}  as_of_date={AS_OF_DATE}  players={N_PLAYERS:,}")

In [0]:
# ---------- Games (6 fictional Emberline titles) ----------
GAMES = [
    ("G01", "Iron Vanguard",    "Battle Royale Shooter", ["PC", "PS5", "Xbox"], "free_to_play",            "2021-03-15"),
    ("G02", "Starfall Legends", "Hero Shooter",          ["PC", "PS5", "Xbox"], "free_to_play",            "2024-09-10"),
    ("G03", "Realm of Ash",     "MMORPG",                ["PC"],                "subscription_plus_store", "2019-06-01"),
    ("G04", "Hollow Crown",     "Action RPG",            ["PC", "PS5", "Xbox"], "premium_plus_dlc",        "2023-02-20"),
    ("G05", "Gem Tide",         "Match-3 Puzzle",        ["iOS", "Android"],    "free_to_play",            "2022-05-05"),
    ("G06", "Pocket Racers",    "Kart Racing",           ["iOS", "Android"],    "free_to_play",            "2025-04-15"),
]
GAME_IDS = [g[0] for g in GAMES]
TITLES = {g[0]: g[1] for g in GAMES}
LAUNCH = {g[0]: g[5] for g in GAMES}
SHOOTERS, MOBILE = ["G01", "G02"], ["G05", "G06"]
LAUNCH_MAP = F.create_map(*[x for k, v in LAUNCH.items() for x in (F.lit(k), lit_date(v))])

games_df = (spark.createDataFrame(GAMES, "game_id STRING, title STRING, genre STRING, platforms ARRAY<STRING>, monetization_model STRING, launch_date STRING")
            .withColumn("launch_date", F.to_date("launch_date"))
            .withColumn("publisher", F.lit("Emberline Studios")))

# ---------- Geography: (country, region, currency, player weight %, units per USD) ----------
GEO = [
    ("US", "North America", "USD", 30, 1.0),   ("CA", "North America", "CAD", 4, 1.37),
    ("MX", "Latin America", "MXN", 4, 18.5),   ("BR", "Latin America", "BRL", 7, 5.4),
    ("AR", "Latin America", "ARS", 2, 950.0),  ("GB", "Europe", "GBP", 8, 0.79),
    ("DE", "Europe", "EUR", 7, 0.92),          ("FR", "Europe", "EUR", 6, 0.92),
    ("ES", "Europe", "EUR", 4, 0.92),          ("IT", "Europe", "EUR", 3, 0.92),
    ("PL", "Europe", "PLN", 3, 3.95),          ("SE", "Europe", "SEK", 2, 10.6),
    ("NL", "Europe", "EUR", 2, 0.92),          ("JP", "Asia Pacific", "JPY", 6, 150.0),
    ("KR", "Asia Pacific", "KRW", 4, 1350.0),  ("AU", "Asia Pacific", "AUD", 4, 1.52),
    ("IN", "Asia Pacific", "INR", 3, 83.0),    ("PH", "Asia Pacific", "PHP", 1, 56.0),
]
COUNTRIES = [g[0] for g in GEO]
CURRENCY_MAP = kv_map({g[0]: g[2] for g in GEO})
FX_MAP = kv_map({g[0]: g[4] for g in GEO})

# ---------- Live-ops calendar (hand-built: real Black Friday / holiday dates matter) ----------
W_START, W_END = date.fromisoformat(START_DATE), date.fromisoformat(END_DATE)
_events = []


def add_event(game_id, name, etype, start, days, discount=0):
    """Append one live-ops event if it overlaps the data window and starts on/after the title's launch."""
    end = start + timedelta(days=days - 1)
    if start > W_END or end < W_START or start < date.fromisoformat(LAUNCH[game_id]):
        return
    _events.append((game_id, name, etype, max(start, W_START), min(end, W_END), discount))


for gi, gid in enumerate(GAME_IDS):
    for y in (2025, 2026):
        if gid in ("G01", "G02", "G04", "G06"):                      # competitive titles: quarterly ranked launch fortnight
            for q, m in enumerate((1, 4, 7, 10), start=1):
                add_event(gid, f"{TITLES[gid]} {y} Ranked Season {q}", "ranked_season", date(y, m, 10), 14)
        for m in range(1, 13):                                          # skin drop every other month
            if (m + gi) % 2 == 0:
                add_event(gid, f"{TITLES[gid]} {date(y, m, 1):%B} {y} Skin Drop", "skin_drop", date(y, m, 18), 7)
        add_event(gid, f"{TITLES[gid]} Summer Sale {y}", "sale", date(y, 7, 1), 10, 25)
        add_event(gid, f"{TITLES[gid]} Black Friday {y}", "sale", date(2025, 11, 28) if y == 2025 else date(2026, 11, 27), 4, 40)
        add_event(gid, f"{TITLES[gid]} Holiday Sale {y}", "sale", date(y, 12, 20), 13, 30)
for gid, nm, spring, finals in [("G01", "Vanguard Championship", (6, 5), (11, 6)), ("G02", "Starfall Masters", (5, 15), (9, 25))]:
    for y in (2025, 2026):
        add_event(gid, f"{nm} {y} Spring Split", "esports", date(y, *spring), 4)
        add_event(gid, f"{nm} {y} Finals", "esports", date(y, *finals), 4)

_events.sort(key=lambda e: (e[3], e[0]))
events_df = spark.createDataFrame(
    [(f"E{i:04d}", *e) for i, e in enumerate(_events, start=1)],
    "event_id STRING, game_id STRING, event_name STRING, event_type STRING, start_date DATE, end_date DATE, discount_pct INT")

# One event per (game, day) for joins: sale > esports > ranked_season > skin_drop
PRIORITY = {"sale": 1, "esports": 2, "ranked_season": 3, "skin_drop": 4}
event_days = materialize(
    events_df.select("game_id", "event_id", "event_type",
                     F.explode(F.sequence("start_date", "end_date")).alias("session_date"),
                     F.element_at(kv_map(PRIORITY), c("event_type")).alias("prio"))
             .withColumn("rn", F.row_number().over(Window.partitionBy("game_id", "session_date").orderBy("prio")))
             .filter("rn = 1").drop("rn", "prio"),
    "event_days")

print(f"games={games_df.count()}  live_ops_events={events_df.count()}  event_days={event_days.count()}")
events_df.groupBy("event_type").count().orderBy("event_type").show()
games_df.show(truncate=False)

In [0]:
CHANNELS = ["organic", "paid_social", "influencer", "cross_promo", "store_feature"]
TIERS = ["whale", "dolphin", "minnow", "non_payer"]
AGE_BANDS = {"13-17": (13, 17), "18-24": (18, 24), "25-34": (25, 34), "35-44": (35, 44), "45+": (45, 60)}
ADJ = ["Shadow", "Iron", "Crimson", "Silent", "Frost", "Ember", "Void", "Lucky",
       "Rapid", "Neon", "Storm", "Ghost", "Atomic", "Turbo", "Mystic", "Feral"]
NOUN = ["Wolf", "Falcon", "Viper", "Knight", "Raven", "Titan", "Fox", "Blade",
        "Ranger", "Phoenix", "Golem", "Racer", "Wizard", "Hunter", "Drake", "Comet"]
DOMAINS = ["example.com", "example.net", "example.org", "mail.test"]   # reserved / fictional domains only
PRE_WINDOW_SHARE = {"G01": 0.55, "G03": 0.80, "G04": 0.50, "G05": 0.50}  # share registered before 2025
RING_START = N_PLAYERS - N_RINGS * RING_SIZE                            # last 600 ids are fraud-ring accounts

# 1) Independent attributes: dbldatagen
# randomSeedMethod="hash_fieldname" gives every column its own random stream. Without it, the columns are
# perfectly correlated (all whales land in the 13-17 band and in the US).
spec = (
    dg.DataGenerator(spark, name="players", rows=N_PLAYERS, partitions=8, randomSeed=SEED, randomSeedMethod="hash_fieldname")
    .withIdOutput()
    .withColumn("country", "string", values=COUNTRIES, weights=[g[3] for g in GEO], random=True)
    .withColumn("acquisition_channel", "string", values=CHANNELS, weights=[40, 25, 15, 12, 8], random=True)
    .withColumn("signup_game_id", "string", values=GAME_IDS, weights=[25, 22, 12, 13, 16, 12], random=True)
    .withColumn("spend_tier", "string", values=TIERS, weights=[2, 8, 20, 70], random=True)
    .withColumn("age_band", "string", values=list(AGE_BANDS), weights=[12, 30, 33, 17, 8], random=True)
    .withColumn("activity_level", "string", values=["casual", "regular", "core"], weights=[50, 35, 15], random=True)
    .withColumn("marketing_opt_in", "int", values=[1, 0], weights=[35, 65], random=True)
)
base = spec.build()

# 2) Latent traits and dependent attributes: PySpark
p = with_uniforms(base.withColumn("p_num", (c("id") + 1).cast("int")), [f"u{i}" for i in range(1, 17)], 100)
ring = c("p_num") > RING_START
age_lo = kv_map({k: lo for k, (lo, hi) in AGE_BANDS.items()})
age_span = kv_map({k: hi - lo + 1 for k, (lo, hi) in AGE_BANDS.items()})

p = (p
    .withColumn("player_id", F.format_string("P%06d", c("p_num")))
    .withColumn("is_fraud_ring", ring)
    .withColumn("ring_id", F.when(ring, (F.floor((c("p_num") - RING_START - 1) / RING_SIZE) + 1).cast("int")))
    .withColumn("signup_game_id", F.when(ring, F.when(c("ring_id") % 2 == 0, F.lit("G01")).otherwise(F.lit("G02")))
                                   .otherwise(c("signup_game_id")))
    .withColumn("age", F.when(ring, 22 + F.floor(c("u1") * 15))
                        .otherwise(F.element_at(age_lo, c("age_band")) + F.floor(c("u1") * F.element_at(age_span, c("age_band"))))
                        .cast("int"))
    .withColumn("date_of_birth", F.date_sub(lit_date(AS_OF_DATE), (c("age") * 365 + F.floor(c("u2") * 365)).cast("int")))
    .withColumn("spend_tier", F.when(ring, "fraud")
                               .when((c("age") < 18) & (c("spend_tier") == "whale"), "minnow")
                               .otherwise(c("spend_tier")))
    .withColumn("activity_level", F.when(ring, "core")
                                   .when((c("spend_tier") == "whale") & (c("u3") < 0.60), "core")
                                   .when((c("spend_tier") == "dolphin") & (c("u3") < 0.35), "core")
                                   .otherwise(c("activity_level")))
    .withColumn("is_toxic", ~ring & (c("u4") < 0.04))                                     # ~4% drive abuse
    .withColumn("is_victim", ~ring & (c("u5") < 0.10))                                    # players they target
    .withColumn("is_cheater", ~ring & c("signup_game_id").isin(SHOOTERS) & (c("u6") < 0.015))
    .withColumn("has_bad_matchmaking", c("u7") < 0.15)
)

# Registration: launch cohorts front-loaded; Realm of Ash mostly pre-window (declining); rings register in bursts
reg_end = lit_date("2026-11-30")
l06, l02 = lit_date(LAUNCH["G06"]), lit_date(LAUNCH["G02"])
pre_share = F.coalesce(F.element_at(kv_map(PRE_WINDOW_SHARE), c("signup_game_id")), F.lit(0.5))
pre_lo = F.greatest(F.element_at(LAUNCH_MAP, c("signup_game_id")), lit_date("2020-01-01"))
reg = (F.when(ring, F.date_add(lit_date(START_DATE), ((c("ring_id") - 1) * 9 + 5 + F.floor(c("u8") * 7)).cast("int")))
        .when(c("signup_game_id") == "G06", F.date_add(l06, F.floor(F.pow(c("u8"), 2) * F.datediff(reg_end, l06)).cast("int")))
        .when(c("signup_game_id") == "G02", F.date_add(l02, F.floor(F.pow(c("u8"), 1.3) * F.datediff(reg_end, l02)).cast("int")))
        .when(c("u9") < pre_share, F.date_add(pre_lo, F.floor(c("u8") * F.datediff(lit_date("2024-12-31"), pre_lo)).cast("int")))
        .otherwise(F.date_add(lit_date(START_DATE), F.floor(c("u8") * F.datediff(reg_end, lit_date(START_DATE))).cast("int"))))
p = p.withColumn("registration_date", reg).withColumn("active_start", F.greatest(c("registration_date"), lit_date(START_DATE)))

# Churn: exponential lifetime; victims, bad matchmaking and new players churn faster, whales slowest
mean_life = (F.lit(900.0)
    * F.when(c("spend_tier") == "whale", 3.0).when(c("spend_tier") == "dolphin", 1.8).when(c("spend_tier") == "minnow", 1.2).otherwise(1.0)
    * F.when(c("activity_level") == "core", 1.5).when(c("activity_level") == "casual", 0.7).otherwise(1.0)
    * F.when(c("registration_date") >= lit_date(START_DATE), 0.55).otherwise(1.0)
    * F.when(c("is_victim"), 0.5).otherwise(1.0)
    * F.when(c("has_bad_matchmaking"), 0.7).otherwise(1.0))
churn_raw = (F.when(ring, F.date_add(c("registration_date"), (30 + F.floor(c("u11") * 60)).cast("int")))
              .otherwise(F.date_add(c("active_start"), F.floor(-F.log(1 - c("u10")) * mean_life).cast("int"))))
p = (p.withColumn("_churn", churn_raw)
      .withColumn("is_churned", c("_churn") <= lit_date(AS_OF_DATE))
      .withColumn("churn_date", F.when(c("is_churned"), c("_churn")))
      .withColumn("active_end", F.coalesce(c("churn_date"), lit_date(AS_OF_DATE))))

# PII (fictional): gamertag, email on reserved domains, 555 phone numbers
p = (p
    .withColumn("gamertag", F.format_string("%s%s%04d", upick(ADJ, c("u12")), upick(NOUN, c("u13")), F.pmod(c("p_num") * 7919, F.lit(10000))))
    .withColumn("email", F.format_string("%s.%d@%s", F.lower(c("gamertag")), c("p_num"), upick(DOMAINS, c("u14"))))
    .withColumn("phone_number", F.format_string("+1-555-%03d-%04d", F.pmod(c("p_num"), F.lit(1000)), F.pmod(c("p_num") * 37, F.lit(10000))))
    .withColumn("account_status", F.when(c("is_churned") & ~ring & (c("u15") < 0.03), "closed").otherwise("active"))
    .withColumn("marketing_opt_in", c("marketing_opt_in") == 1))

players_full = materialize(p.select(
    "player_id", "p_num", "gamertag", "email", "phone_number", "date_of_birth", "age", "country",
    "registration_date", "acquisition_channel", "signup_game_id", "marketing_opt_in", "account_status",
    "spend_tier", "activity_level", "is_toxic", "is_victim", "is_cheater", "is_fraud_ring", "ring_id",
    "has_bad_matchmaking", "active_start", "active_end", "churn_date", "is_churned"), "players_full")

print(f"players={players_full.count():,}")
players_full.groupBy("spend_tier").count().orderBy("spend_tier").show()
players_full.agg(
    F.sum(c("is_fraud_ring").cast("int")).alias("ring_accounts"),
    F.countDistinct("ring_id").alias("rings"),
    F.sum(c("is_toxic").cast("int")).alias("toxic"),
    F.sum(c("is_victim").cast("int")).alias("victims"),
    F.sum(c("is_cheater").cast("int")).alias("cheaters"),
    F.round(F.avg(c("is_churned").cast("int")) * 100, 1).alias("churned_pct"),
    F.sum((c("age") < 18).cast("int")).alias("under_18")).show()
players_full.groupBy(F.year("registration_date").alias("reg_year")).count().orderBy("reg_year").show()
players_full.select("player_id", "gamertag", "email", "phone_number", "date_of_birth", "country", "registration_date", "signup_game_id").show(5, truncate=False)

In [0]:
MODELS = {
    "PC": ["Custom Gaming PC", "Gaming Laptop", "Steam Deck"],
    "PS5": ["PlayStation 5", "PlayStation 5 Pro"],
    "Xbox": ["Xbox Series X", "Xbox Series S"],
    "iOS": ["iPhone 15", "iPhone 14", "iPhone 13", "iPad Air"],
    "Android": ["Galaxy S24", "Pixel 8", "Redmi Note 13", "Galaxy A54"],
}

dv = with_uniforms(players_full.select("player_id", "p_num", "signup_game_id", "is_fraud_ring", "ring_id", "registration_date"),
                   ["v1", "v2"], 200)
dv = (dv.withColumn("n_devices", F.when(c("is_fraud_ring"), F.lit(1))
                                  .otherwise(1 + (c("v1") < 0.35).cast("int") + (c("v2") < 0.10).cast("int")))
        .withColumn("device_n", F.explode(F.sequence(F.lit(1), c("n_devices")))))
dv = with_uniforms(dv, ["w1", "w2", "w3", "w4", "w5"], 210)

platform = (F.when(c("is_fraud_ring"), F.when(c("w1") < 0.7, F.lit("Android")).otherwise(F.lit("PC")))
             .when((c("device_n") == 1) & c("signup_game_id").isin(MOBILE), wpick(["iOS", "Android"], [45, 55], c("w1")))
             .when(c("device_n") == 1, wpick(["PC", "PS5", "Xbox"], [50, 30, 20], c("w1")))
             .otherwise(wpick(["iOS", "Android", "PC", "PS5", "Xbox"], [30, 30, 15, 15, 10], c("w1"))))
dv = dv.withColumn("platform", platform).withColumn("device_id", F.format_string("%s-D%d", c("player_id"), c("device_n")))

is_emulator = F.when(c("is_fraud_ring"), (c("platform") == "Android") & (c("w2") < 0.85)).otherwise(c("w2") < 0.004)
household = (~c("is_fraud_ring")) & (c("device_n") == 1) & (F.pmod(F.xxhash64(F.floor(c("p_num") / 2)), F.lit(100)) < 3)
fingerprint = (F.when(c("is_fraud_ring"), F.sha2(F.format_string("ring-%d-%d", c("ring_id"), F.pmod(c("p_num"), F.lit(2))), 256))
                .when(household, F.sha2(F.format_string("hh-%d", F.floor(c("p_num") / 2)), 256))   # legit shared family devices
                .otherwise(F.sha2(F.concat(F.lit("dev-"), c("device_id")), 256)))
model = F.when(c("platform") == "PC", upick(MODELS["PC"], c("w4")))
for plat in ["PS5", "Xbox", "iOS", "Android"]:
    model = model.when(c("platform") == plat, upick(MODELS[plat], c("w4")))

devices_full = materialize(dv
    .withColumn("is_emulator", is_emulator)
    .withColumn("is_rooted_or_jailbroken", F.when(c("is_fraud_ring"), c("w3") < 0.4)
                                           .when(c("platform").isin("iOS", "Android"), c("w3") < 0.02).otherwise(F.lit(False)))
    .withColumn("device_type", F.when(c("platform") == "PC", "pc").when(c("platform").isin("PS5", "Xbox"), "console").otherwise("mobile"))
    .withColumn("device_model", F.when(c("is_emulator"), F.lit("Generic x86_64 Emulator")).otherwise(model))
    .withColumn("device_fingerprint", F.substring(fingerprint, 1, 32))
    .withColumn("first_seen_date", F.least(F.date_add(c("registration_date"), ((c("device_n") - 1) * F.floor(c("w5") * 200)).cast("int")),
                                           lit_date(AS_OF_DATE)))
    .select("device_id", "player_id", "device_n", "device_type", "platform", "device_model", "device_fingerprint",
            "is_emulator", "is_rooted_or_jailbroken", "first_seen_date", "is_fraud_ring"), "devices_full")

fp_share = devices_full.groupBy("device_fingerprint").agg(F.countDistinct("player_id").alias("accounts"),
                                                          F.max(c("is_fraud_ring").cast("int")).alias("ring"))
print(f"devices={devices_full.count():,}")
devices_full.groupBy("platform").count().orderBy("platform").show()
devices_full.groupBy("is_fraud_ring").agg(F.count("*").alias("devices"),
    F.round(F.avg(c("is_emulator").cast("int")) * 100, 1).alias("emulator_pct"),
    F.round(F.avg(c("is_rooted_or_jailbroken").cast("int")) * 100, 1).alias("rooted_pct")).show()
print("Fingerprints shared by 2+ accounts (ring vs household):")
fp_share.filter("accounts >= 2").groupBy("ring").agg(F.count("*").alias("fingerprints"), F.avg("accounts").alias("avg_accounts")).show()

In [0]:
HOUR_W = [3, 2, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 6, 7, 9, 10, 10, 9, 7, 5]   # evening peak
HOUR_BAG = [h for h, w in enumerate(HOUR_W) for _ in range(w)]

n_dev = devices_full.groupBy("player_id").agg(F.count("*").alias("n_devices"))
ev = F.broadcast(event_days)

# 1) One row per player per day of their active life
pdays = (players_full.join(n_dev, "player_id")
    .select("player_id", "p_num", "signup_game_id", "activity_level", "spend_tier", "is_cheater", "is_toxic",
            "is_fraud_ring", "has_bad_matchmaking", "country", "registration_date", "churn_date",
            "active_start", "active_end", "n_devices")
    .withColumn("session_date", F.explode(F.sequence("active_start", "active_end")))
    .join(ev.select(c("game_id").alias("signup_game_id"), "session_date", c("event_type").alias("primary_event_type")),
          ["signup_game_id", "session_date"], "left"))

# 2) Probability of playing that day
t = F.datediff(c("session_date"), lit_date(START_DATE)) / 730.0
dow, mon, dom = F.dayofweek("session_date"), F.month("session_date"), F.dayofmonth("session_date")
p_base = (F.when(c("is_fraud_ring"), 0.8).when(c("activity_level") == "core", 0.50)
           .when(c("activity_level") == "regular", 0.26).otherwise(0.095))
season = (F.when(dow.isin(1, 7), 1.3).otherwise(1.0)                                         # weekend uplift
          * F.when(mon.isin(6, 7, 8), 1.1).when((mon == 9) & (dom <= 21), 0.85).otherwise(1.0)  # summer / back-to-school
          * F.when(((mon == 12) & (dom >= 20)) | ((mon == 1) & (dom <= 3)), 1.25).otherwise(1.0))
trend = (F.when(c("signup_game_id") == "G03", 1.0 - 0.4 * t)        # Realm of Ash declining
          .when(c("signup_game_id") == "G02", 0.85 + 0.4 * t)       # Starfall Legends growing
          .when(c("signup_game_id") == "G06", 0.9 + 0.3 * t)         # Pocket Racers growing
          .otherwise(1.0))
decay = F.when(c("churn_date").isNotNull(),
               F.greatest(F.lit(0.1), F.least(F.lit(1.0), F.datediff(c("churn_date"), c("session_date")) / 30.0))).otherwise(1.0)
event_up = (F.when(c("primary_event_type").isin("ranked_season", "esports"), 1.25)
             .when(c("primary_event_type") == "sale", 1.1).when(c("primary_event_type") == "skin_drop", 1.05).otherwise(1.0))
p_play = F.least(F.lit(0.95), p_base * season * trend * decay * event_up)

played = with_uniforms(pdays, ["s0"], 300).filter(c("s0") < p_play)
played = with_uniforms(played, ["s1", "s2", "s3"], 301)
n_sess = (1 + (c("s1") < 0.3).cast("int") + (c("s2") < 0.1).cast("int")
          + ((c("activity_level") == "core") & (c("s3") < 0.3)).cast("int"))
sess = played.withColumn("k", F.explode(F.sequence(F.lit(1), n_sess)))
sess = with_uniforms(sess, [f"g{i}" for i in range(1, 17)], 320)

# 3) Which title: 85% the player's main game, 15% one fixed secondary title per player (never before its launch)
GAME_IDX = kv_map({g: i for i, g in enumerate(GAME_IDS)})
secondary_idx = (F.element_at(GAME_IDX, c("signup_game_id")) + 1 + F.pmod(F.xxhash64(c("p_num"), F.lit("secondary")), F.lit(5))) % 6
other = F.element_at(F.array(*[F.lit(g) for g in GAME_IDS]), (secondary_idx + 1).cast("int"))
sess = sess.withColumn("game_id",
    F.when(c("is_fraud_ring") | (c("g1") < 0.85), c("signup_game_id"))
     .when(F.element_at(LAUNCH_MAP, other) <= c("session_date"), other)
     .otherwise(c("signup_game_id")))
sess = (sess.join(ev.select("game_id", "session_date", c("event_id").alias("live_ops_event_id"), "event_type"),
                  ["game_id", "session_date"], "left"))

# 4) Session metrics
shooter = c("game_id").isin(SHOOTERS)
cheating = shooter & c("is_cheater")
core_mult = F.when(c("activity_level") == "core", 1.2).otherwise(1.0)
duration = (F.when(shooter, 15 + c("g6") * 30).when(c("game_id") == "G03", 30 + c("g6") * 150)
             .when(c("game_id") == "G04", 20 + c("g6") * 100).otherwise(3 + c("g6") * 17)) * core_mult
sess = sess.withColumn("duration_min", F.round(duration, 1))
matches = (F.when(shooter, F.greatest(F.lit(1), F.round(c("duration_min") / 18)))
            .when(c("game_id") == "G06", F.greatest(F.lit(1), F.round(c("duration_min") / 4)))
            .when(c("game_id") == "G05", F.greatest(F.lit(1), F.round(c("duration_min") / 3)))
            .otherwise(F.greatest(F.lit(1), F.floor(c("duration_min") / 30)))).cast("int")
sess = sess.withColumn("matches", matches)
start_secs = upick(HOUR_BAG, c("g3")) * 3600 + F.floor(c("g4") * 60) * 60 + F.floor(c("g5") * 60)
ip_country = (F.when(c("is_fraud_ring") & (c("g14") < 0.7), upick(COUNTRIES, c("g15")))          # VPN mismatch
               .when(~c("is_fraud_ring") & (c("g14") < 0.02), upick(COUNTRIES, c("g15")))          # travel
               .otherwise(c("country")))
ping = (F.when(c("has_bad_matchmaking"), 60 + c("g11") * 120).otherwise(20 + c("g11") * 60)
        + F.when(c("is_fraud_ring"), 80).otherwise(0) + F.when(c("game_id").isin(MOBILE), 20).otherwise(0))
mmq = F.when(c("game_id") == "G05", F.lit(None).cast("double")).otherwise(
      F.round(F.when(c("has_bad_matchmaking"), 0.15 + c("g12") * 0.45).otherwise(0.6 + c("g12") * 0.4), 2))

sess = (sess
    .withColumn("session_id", F.format_string("S%s%06d%d", F.date_format("session_date", "yyyyMMdd"), c("p_num"), c("k")))
    .withColumn("device_id", F.format_string("%s-D%d", c("player_id"), (F.floor(c("g13") * c("n_devices")) + 1).cast("int")))
    .withColumn("session_start_ts", ts_plus_seconds(c("session_date"), start_secs))
    .withColumn("session_end_ts", ts_plus_seconds(c("session_start_ts"), c("duration_min") * 60))
    .withColumn("kills", F.when(shooter, F.round(c("matches") * (2 + c("g7") * 6) * F.when(cheating, 2.5).otherwise(1.0)).cast("int")))
    .withColumn("deaths", F.when(shooter, F.round(c("matches") * (2 + c("g8") * 5) * F.when(cheating, 0.4).otherwise(1.0)).cast("int")))
    .withColumn("accuracy_pct", F.when(shooter, F.round(F.when(cheating, 55 + c("g9") * 30)
                                                        .otherwise(15 + c("g9") * 20 + F.when(c("activity_level") == "core", 5).otherwise(0)), 1)))
    .withColumn("headshot_pct", F.when(shooter, F.round(F.when(cheating, 55 + c("g10") * 35).otherwise(8 + c("g10") * 20), 1)))
    .withColumn("ping_ms", F.round(ping).cast("int"))
    .withColumn("matchmaking_quality", mmq)
    .withColumn("ip_country", ip_country))

sessions_full = materialize(sess.select(
    "session_id", "player_id", "p_num", "game_id", "device_id", "session_date", "session_start_ts", "session_end_ts",
    "duration_min", "matches", "kills", "deaths", "accuracy_pct", "headshot_pct", "ping_ms", "matchmaking_quality",
    "ip_country", "live_ops_event_id", "event_type",
    # latent columns (used by later generators, never written to raw)
    "spend_tier", "activity_level", "is_cheater", "is_toxic", "is_fraud_ring", "country", "registration_date"), "sessions_full")

print(f"sessions={sessions_full.count():,}  distinct_ids={sessions_full.select('session_id').distinct().count():,}")
sessions_full.groupBy(F.year("session_date").alias("year")).count().orderBy("year").show()
sessions_full.groupBy("game_id").agg(F.count("*").alias("sessions"), F.countDistinct("player_id").alias("players"),
                                     F.round(F.avg("duration_min"), 1).alias("avg_min")).orderBy("game_id").show()

In [0]:
PRICES = [0.99, 4.99, 9.99, 19.99, 49.99, 99.99]
PRICE_W = {"whale": [5, 10, 20, 25, 25, 15], "dolphin": [10, 25, 30, 20, 12, 3], "minnow": [40, 35, 18, 6, 1, 0]}
STOLEN_CARD_COUNTRIES = ["US", "GB", "CA", "DE", "AU"]
s = sessions_full
tier = c("spend_tier")
mon, dom = F.month("session_date"), F.dayofmonth("session_date")
black_friday = c("session_date").between("2025-11-28", "2025-12-01") | c("session_date").between("2026-11-27", "2026-11-30")

# ---------- Normal payers: purchases happen inside sessions ----------
buy_mult = (F.when(c("event_type") == "sale", 1.8).when(c("event_type") == "skin_drop", 1.5).otherwise(1.0)
            * F.when(black_friday, 2.0).otherwise(1.0)
            * F.when((mon == 12) & (dom >= 15), 1.4).otherwise(1.0)
            * F.when(mon.isin(6, 7, 8), 1.1).otherwise(1.0))
p_buy = F.when(tier == "whale", 0.68).when(tier == "dolphin", 0.35).when(tier == "minnow", 0.09).otherwise(0.0) * buy_mult
normal = with_uniforms(s.filter(tier.isin("whale", "dolphin", "minnow")), ["b0"], 400).filter(c("b0") < F.least(F.lit(0.95), p_buy))
normal = with_uniforms(normal, ["b1", "b2"], 401)
normal = normal.withColumn("k2", F.explode(F.sequence(F.lit(1),
    1 + ((tier == "whale") & (c("b1") < 0.4)).cast("int") + ((tier == "whale") & (c("b2") < 0.15)).cast("int"))))
normal = with_uniforms(normal, [f"t{i}" for i in range(1, 8)], 410)
item = (F.when(c("game_id").isin(SHOOTERS), wpick(["skin", "currency_pack", "battle_pass", "bundle"], [45, 30, 15, 10], c("t2")))
         .when(c("game_id") == "G03", wpick(["subscription", "currency_pack", "cosmetic"], [40, 30, 30], c("t2")))
         .when(c("game_id") == "G04", wpick(["dlc", "currency_pack", "skin"], [30, 30, 40], c("t2")))
         .when(c("game_id") == "G05", wpick(["currency_pack", "booster", "bundle"], [50, 40, 10], c("t2")))
         .otherwise(wpick(["currency_pack", "kart_skin", "battle_pass"], [40, 40, 20], c("t2"))))
price = (F.when(tier == "whale", wpick(PRICES, PRICE_W["whale"], c("t1")))
          .when(tier == "dolphin", wpick(PRICES, PRICE_W["dolphin"], c("t1")))
          .otherwise(wpick(PRICES, PRICE_W["minnow"], c("t1"))))
normal = (normal.withColumn("item_type", item)
    .withColumn("amount_usd", F.when(c("item_type") == "battle_pass", 9.99).when(c("item_type") == "subscription", 14.99)
                               .when(c("item_type") == "dlc", 19.99).otherwise(price))
    .withColumn("payment_method", F.when(c("game_id").isin(MOBILE), wpick(["wallet", "card", "paypal", "gift_card"], [80, 12, 5, 3], c("t3")))
                                   .otherwise(wpick(["card", "paypal", "wallet", "gift_card"], [55, 20, 20, 5], c("t3"))))
    .withColumn("billing_country", F.when(c("t4") < 0.03, upick(COUNTRIES, c("t5"))).otherwise(c("country"))))

# ---------- Fraud rings: bursts of $50-$100 currency packs on stolen cards in the first 3 weeks ----------
rings = s.filter(c("is_fraud_ring") & (F.datediff("session_date", "registration_date") <= 21))
rings = with_uniforms(rings, ["r0"], 450).filter(c("r0") < 0.25)
rings = with_uniforms(rings, ["r1", "r2"], 451)
rings = rings.withColumn("k2", F.explode(F.sequence(F.lit(1), 1 + (c("r1") < 0.3).cast("int"))))
rings = with_uniforms(rings, [f"t{i}" for i in range(1, 8)], 460)
rings = (rings.withColumn("item_type", F.lit("currency_pack"))
    .withColumn("amount_usd", F.when(c("t1") < 0.55, 49.99).otherwise(99.99))
    .withColumn("payment_method", F.when(c("t3") < 0.95, F.lit("card")).otherwise(F.lit("gift_card")))
    .withColumn("billing_country", upick(STOLEN_CARD_COUNTRIES, c("t5"))))

txn_cols = ["session_id", "player_id", "p_num", "game_id", "device_id", "session_start_ts", "duration_min", "k2", "t6",
            "item_type", "amount_usd", "payment_method", "billing_country", "ip_country", "live_ops_event_id",
            "event_type", "spend_tier", "is_fraud_ring"]
tx = normal.select(*txn_cols).unionByName(rings.select(*txn_cols))
transactions_full = materialize(tx
    .withColumn("txn_id", F.format_string("T%s%d", F.substring("session_id", 2, 40), c("k2")))
    .withColumn("txn_ts", ts_plus_seconds(c("session_start_ts"), F.floor(c("t6") * c("duration_min") * 60)))
    .withColumn("txn_date", F.to_date("txn_ts"))
    .withColumn("currency", F.element_at(CURRENCY_MAP, c("billing_country")))
    .withColumn("amount_local", F.round(c("amount_usd") * F.element_at(FX_MAP, c("billing_country")), 2))
    .filter(c("txn_ts") < F.lit("2027-01-01").cast("timestamp"))
    .select("txn_id", "player_id", "p_num", "game_id", "device_id", "session_id", "txn_ts", "txn_date", "item_type",
            "amount_usd", "currency", "amount_local", "payment_method", "billing_country", "ip_country",
            "live_ops_event_id", "event_type", "spend_tier", "is_fraud_ring"), "transactions_full")

# ---------- Chargebacks: risk-factor driven, arrive 15-75 days after purchase ----------
feat = (transactions_full
    .join(players_full.select("player_id", "date_of_birth", "registration_date"), "player_id")
    .join(devices_full.select("device_id", "is_emulator"), "device_id", "left")
    .withColumn("is_minor", F.floor(F.months_between("txn_date", "date_of_birth") / 12) < 18)
    .withColumn("geo_mismatch", c("billing_country") != c("ip_country"))
    .withColumn("new_account", F.datediff("txn_date", "registration_date") < 30))
p_cb = F.when(c("is_fraud_ring"), F.lit(0.65)).otherwise(F.least(F.lit(0.5), F.lit(0.003)
    * F.when(c("is_minor"), 8.0).otherwise(1.0)            # friendly fraud: kids dispute parents' cards
    * F.when(c("geo_mismatch"), 3.0).otherwise(1.0)
    * F.when(c("is_emulator"), 4.0).otherwise(1.0)
    * F.when(c("new_account"), 2.0).otherwise(1.0)))
cb = with_uniforms(feat, ["cb0", "cb1", "cb2", "cb3"], 500).filter(c("cb0") < p_cb)
chargebacks_full = materialize(cb
    .withColumn("chargeback_id", F.format_string("CB%s", F.substring("txn_id", 2, 40)))
    .withColumn("chargeback_date", F.date_add("txn_date", (15 + F.floor(c("cb1") * 61)).cast("int")))
    .withColumn("reason_code", F.when(c("is_fraud_ring"), F.lit("FRAUD_CARD_NOT_PRESENT"))
                                .when(c("is_minor") & (c("cb2") < 0.8), F.lit("UNRECOGNIZED_TRANSACTION"))
                                .otherwise(wpick(["ITEM_NOT_RECEIVED", "DUPLICATE_CHARGE", "UNRECOGNIZED_TRANSACTION"], [40, 20, 40], c("cb2"))))
    .withColumn("fraud_type", F.when(c("is_fraud_ring"), "fraud_ring").when(c("is_minor"), "friendly_fraud").otherwise("other_dispute"))
    .withColumn("card_network", F.when(c("payment_method") == "card", wpick(["visa", "mastercard", "amex"], [55, 35, 10], c("cb3")))
                                 .when(c("payment_method") == "paypal", "paypal").when(c("payment_method") == "wallet", "app_store")
                                 .otherwise("gift_card_issuer"))
    .withColumn("disputed_amount_usd", c("amount_usd"))
    .withColumn("chargeback_fee_usd", F.lit(15.0))
    .withColumn("is_visible_by_as_of", c("chargeback_date") <= lit_date(AS_OF_DATE))
    .select("chargeback_id", "txn_id", "chargeback_date", "reason_code", "disputed_amount_usd", "chargeback_fee_usd",
            "card_network", "fraud_type", "is_visible_by_as_of"), "chargebacks_full")

print(f"transactions={transactions_full.count():,}  distinct_ids={transactions_full.select('txn_id').distinct().count():,}")
transactions_full.groupBy("spend_tier").agg(F.count("*").alias("txns"), F.round(F.sum("amount_usd"), 0).alias("revenue_usd")).orderBy("spend_tier").show()
chargebacks_full.groupBy("fraud_type", "is_visible_by_as_of").count().orderBy("fraud_type", "is_visible_by_as_of").show()
print(f"chargebacks={chargebacks_full.count():,}  min_lag_days / max_lag_days:")
chargebacks_full.join(transactions_full.select("txn_id", "txn_date"), "txn_id").agg(
    F.min(F.datediff("chargeback_date", "txn_date")).alias("min_lag"), F.max(F.datediff("chargeback_date", "txn_date")).alias("max_lag")).show()

In [0]:
TEMPLATES = {
    "HARASSMENT": ["you are so bad, just uninstall the game", "nobody wants you on this team, quit already",
                   "worst player I have ever seen, go back to the tutorial", "you are a waste of a slot, leave the match",
                   "everyone report this useless player", "keep crying, loser", "you're pathetic, get off my team",
                   "go touch grass you worthless noob"],
    "HATE_SPEECH": ["go back to your country you [slur]", "[slur] like you should not be allowed to play",
                    "typical [slur], ruining every match", "no [slur] allowed in this lobby", "I hate [slur] players, get out"],
    "THREAT": ["I know where you live, watch yourself", "I will find you after this match", "say that again and I will hurt you",
               "I am going to dox you tonight", "you better sleep with one eye open"],
    "SCAM": ["selling 10000 Embers for $15, DM me", "free legendary skins at ember-gifts dot net, just log in",
             "cheap Embers 70% off, pay with gift card", "rank boost service cheap, send me your account password",
             "giving away 5000 Embers, verify your account at embergift dot shop"],
    "BANTER": ["lol you are trash at this, gg rematch?", "get rekt noob haha, nice try though", "I am gonna kill you next round haha",
               "you absolute idiot, that was hilarious", "that headshot was sick, you monster", "stop camping you coward lol",
               "you loser, buy me a coffee for that carry"],
    "NEUTRAL": ["gg", "nice shot!", "need a healer on B", "anyone want to queue again?", "push mid", "thanks for the carry",
                "lag is bad tonight", "glhf", "regroup at spawn", "nice drift!", "one more level to go", "who has the objective?",
                "good game everyone", "wait for me at the gate", "can someone share a booster?", "on my way"],
}
TOXIC_CATS = ["HARASSMENT", "HATE_SPEECH", "THREAT"]

s = sessions_full.withColumnRenamed("is_toxic", "sender_is_toxic")
p_chat = (F.when(c("is_fraud_ring"), 0.45).when(c("sender_is_toxic"), 0.5)
           .when(c("game_id") == "G05", 0.05).otherwise(0.18))
ch = with_uniforms(s, ["m0"], 600).filter(c("m0") < p_chat)
ch = with_uniforms(ch, ["m1", "m2"], 601)
ch = ch.withColumn("k3", F.explode(F.sequence(F.lit(1), 1 + (c("m1") < 0.45).cast("int") + (c("m2") < 0.2).cast("int"))))
ch = with_uniforms(ch, [f"x{i}" for i in range(1, 9)], 610)

event_mult = F.when(c("event_type").isin("ranked_season", "esports"), 1.5).otherwise(1.0)   # competitive events raise toxicity
p_tox = F.when(c("sender_is_toxic"), 0.35).otherwise(0.004) * event_mult
p_ban = F.when(c("sender_is_toxic"), 0.08).otherwise(0.03)
tox_sub = F.when(c("sender_is_toxic"), wpick(TOXIC_CATS, [60, 22, 18], c("x2"))).otherwise(wpick(["HARASSMENT", "THREAT"], [85, 15], c("x2")))
category = (F.when(c("is_fraud_ring"), F.when(c("x1") < 0.6, F.lit("SCAM")).otherwise(F.lit("NEUTRAL")))
             .when(c("x1") < p_tox, tox_sub)
             .when(c("x1") < p_tox + p_ban, F.lit("BANTER"))
             .otherwise(F.lit("NEUTRAL")))
ch = ch.withColumn("category", category)
text = None
for cat, msgs in TEMPLATES.items():
    text = (F.when(c("category") == cat, upick(msgs, c("x3"))) if text is None
            else text.when(c("category") == cat, upick(msgs, c("x3"))))
ch = (ch.withColumn("is_obfuscated", c("category").isin(*TOXIC_CATS) & (c("x4") < 0.3))          # leetspeak evasion
        .withColumn("message_text", F.when(c("is_obfuscated"), F.translate(F.lower(text), "aeiost", "4310$7")).otherwise(text))
        .withColumn("message_id", F.format_string("M%s%d", F.substring("session_id", 2, 40), c("k3"))))

# Recipients: 80% of abusive messages target the latent victim group
victims = F.broadcast(players_full.filter("is_victim")
                      .select(c("player_id").alias("victim_player_id"), F.row_number().over(Window.orderBy("p_num")).alias("vidx")))
n_victims = players_full.filter("is_victim").count()
ch = ch.withColumn("vidx", (F.pmod(F.xxhash64("message_id"), F.lit(n_victims)) + 1).cast("int")).join(victims, "vidx", "left")
rand_recipient = F.format_string("P%06d", (F.pmod(F.xxhash64("message_id", F.lit("r")), F.lit(N_PLAYERS)) + 1).cast("int"))
recipient = F.when(c("category").isin(*TOXIC_CATS) & (c("x7") < 0.8), c("victim_player_id")).otherwise(rand_recipient)
ch = ch.withColumn("recipient_player_id", recipient)
ch = ch.withColumn("recipient_player_id", F.when(c("recipient_player_id") == c("player_id"),
                                                  F.format_string("P%06d", (F.pmod(c("p_num"), F.lit(N_PLAYERS)) + 1).cast("int")))
                                            .otherwise(c("recipient_player_id")))

chat_full = materialize(ch
    .withColumn("sent_ts", ts_plus_seconds(c("session_start_ts"), F.floor(c("x5") * c("duration_min") * 60)))
    .withColumn("channel", F.when(c("category") == "SCAM", wpick(["global", "dm"], [50, 50], c("x6")))
                            .otherwise(wpick(["match", "team", "dm", "global"], [40, 35, 15, 10], c("x6"))))
    .withColumn("is_toxic", c("category").isin(*TOXIC_CATS, "SCAM"))
    .withColumn("is_banter", c("category") == "BANTER")
    .withColumn("severity", F.when(c("category") == "HARASSMENT", 1 + (c("x8") < 0.4).cast("int"))
                             .when(c("category").isin("HATE_SPEECH", "THREAT"), 3).when(c("category") == "SCAM", 2).otherwise(0))
    .filter(c("sent_ts") < F.lit("2027-01-01").cast("timestamp"))
    .select("message_id", "session_id", "game_id", "channel", c("player_id").alias("sender_player_id"), "recipient_player_id",
            "sent_ts", "message_text", F.lit("en").alias("language"), "category", "is_toxic", "is_banter", "severity",
            "is_obfuscated", "event_type", "sender_is_toxic", "is_fraud_ring"), "chat_full")

print(f"chat_messages={chat_full.count():,}  distinct_ids={chat_full.select('message_id').distinct().count():,}")
chat_full.groupBy("category").agg(F.count("*").alias("messages"), F.sum(c("is_obfuscated").cast("int")).alias("obfuscated")).orderBy("category").show()
chat_full.filter(c("category").isin(*TOXIC_CATS)).select("category", "message_text").show(6, truncate=False)

In [0]:
# ---------- Source 1: recipient reports sender after a toxic message (most direct) ----------
tox_msgs = chat_full.filter(c("is_toxic") & c("recipient_player_id").isNotNull())
tox_reports = with_uniforms(tox_msgs, ["q0"], 700).filter(c("q0") < 0.28)  # ~28% of toxic msgs get reported
tox_reports = with_uniforms(tox_reports, ["q1"], 701)
tox_reports = (tox_reports
    .withColumn("report_id", F.format_string("RPT-M-%s", F.substring("message_id", 2, 40)))
    .withColumn("reporter_player_id", c("recipient_player_id"))
    .withColumn("reported_player_id", c("sender_player_id"))
    .withColumn("report_date", F.to_date("sent_ts"))
    .withColumn("report_category",
        F.when(c("category") == "HATE_SPEECH", F.lit("HATE_SPEECH"))
         .when(c("category") == "THREAT",      F.lit("THREAT"))
         .when(c("category") == "SCAM",        F.lit("SCAM"))
         .otherwise(F.lit("HARASSMENT")))
    .withColumn("evidence_message_id", c("message_id"))
    .withColumn("match_session_id", c("session_id")))

# ---------- Source 2: cheating reports — players notice impossible accuracy ----------
# One cheating report per cheater-session at ~15% probability (another player in the session files it)
cheat_sess = sessions_full.filter("is_cheater").select(
    "session_id", "player_id", "game_id", "session_date", "live_ops_event_id")
cheat_sess = with_uniforms(cheat_sess, ["q0", "q1"], 710)
cheat_reports = cheat_sess.filter(c("q0") < 0.15)
cheat_reports = (
    cheat_reports
    .withColumn("report_id", F.format_string("RPT-C-%s", F.substring("session_id", 2, 40)))
    .withColumn("reporter_player_id",
        F.format_string("P%06d", (F.pmod(F.xxhash64("session_id", F.lit("cheat")), F.lit(N_PLAYERS - 600)) + 1).cast("int")))
    .withColumn("reported_player_id", c("player_id"))
    .withColumn("report_date", c("session_date"))
    .withColumn("report_category", F.lit("CHEATING"))
    .withColumn("evidence_message_id", F.lit(None).cast("string"))
    .withColumn("match_session_id", c("session_id")))

# ---------- Source 3: griefing / other reports at low base rate ----------
base_sess = sessions_full.filter(c("game_id").isin(SHOOTERS + ["G03", "G04"]))
grief_sess = with_uniforms(base_sess, ["q0"], 720).filter(c("q0") < 0.004)
grief_sess = with_uniforms(grief_sess, ["q1", "q2"], 721)
grief_reports = (
    grief_sess
    .withColumn("report_id", F.format_string("RPT-G-%s", F.substring("session_id", 2, 40)))
    .withColumn("reporter_player_id",
        F.format_string("P%06d", (F.pmod(F.xxhash64("session_id", F.lit("grief")), F.lit(N_PLAYERS - 600)) + 1).cast("int")))
    .withColumn("reported_player_id",
        F.format_string("P%06d", (F.pmod(F.xxhash64("session_id", F.lit("target")), F.lit(N_PLAYERS - 600)) + 1).cast("int")))
    .withColumn("report_date", c("session_date"))
    .withColumn("report_category", wpick(["GRIEFING", "OTHER"], [65, 35], c("q1")))
    .withColumn("evidence_message_id", F.lit(None).cast("string"))
    .withColumn("match_session_id", c("session_id")))

RPT_COLS = ["report_id", "reporter_player_id", "reported_player_id", "report_date",
            "report_category", "evidence_message_id", "match_session_id", "game_id"]
reports_full = materialize(
    tox_reports.select(*RPT_COLS)
    .unionByName(cheat_reports.select(*RPT_COLS))
    .unionByName(grief_reports.select(*RPT_COLS))
    .dropDuplicates(["report_id"]),
    "reports_full")

print(f"player_reports={reports_full.count():,}  distinct_ids={reports_full.select('report_id').distinct().count():,}")
reports_full.groupBy("report_category").count().orderBy("report_category").show()
reports_full.groupBy(F.year("report_date").alias("year")).count().orderBy("year").show()

In [0]:
from concurrent.futures import ThreadPoolExecutor

# ─── Column masks: strip latent traits from raw ────────────────────────────
players_raw_df = (
    with_uniforms(players_full, ["dq1"], 903)
    .withColumn("email", F.when(c("dq1") < 0.005, F.lit(None).cast("string")).otherwise(c("email")))
    .select("player_id", "gamertag", "email", "phone_number", "date_of_birth",
            "age", "country", "registration_date", "signup_game_id", "acquisition_channel",
            "account_status", "marketing_opt_in"))

devices_raw_df = devices_full.select(
    "device_id", "player_id", "device_type", "platform", "device_model",
    "device_fingerprint", "is_emulator", "is_rooted_or_jailbroken", "first_seen_date")

sessions_raw_df = (
    with_uniforms(sessions_full, ["dq1", "dq2"], 900)
    .withColumn("duration_min",
        F.when(c("dq1") < 0.003, F.lit(-5.0)).otherwise(c("duration_min")))   # DQ: impossible value
    .withColumn("accuracy_pct",
        F.when((c("dq2") < 0.002) & c("accuracy_pct").isNotNull(), F.lit(105.0))
         .otherwise(c("accuracy_pct")))                                          # DQ: out-of-range
    .select("session_id", "player_id", "game_id", "device_id", "session_date",
            "session_start_ts", "session_end_ts", "duration_min", "matches",
            "kills", "deaths", "accuracy_pct", "headshot_pct", "ping_ms",
            "matchmaking_quality", "ip_country", "live_ops_event_id"))

transactions_raw_df = (
    with_uniforms(transactions_full, ["dq1"], 901)
    .withColumn("payment_method",
        F.when(c("dq1") < 0.004, F.lit("cryptocurrency")).otherwise(c("payment_method")))  # DQ: bad enum
    .select("txn_id", "player_id", "game_id", "device_id", "session_id",
            "txn_ts", "txn_date", "item_type", "amount_usd", "currency",
            "amount_local", "payment_method", "billing_country", "ip_country",
            "live_ops_event_id"))

# Chargebacks: late-arriving — file is keyed on chargeback_date, NOT txn_date
# Only those visible by as_of_date are written; the rest arrive "in the future"
chargebacks_raw_df = (
    chargebacks_full.filter("is_visible_by_as_of")
    .join(transactions_full.select("txn_id", "player_id"), "txn_id")
    .select("chargeback_id", "txn_id", "player_id", "chargeback_date",
            "reason_code", "disputed_amount_usd", "chargeback_fee_usd", "card_network"))

chat_raw_df = (
    chat_full
    .withColumn("sent_date", F.to_date("sent_ts"))
    .select("message_id", "session_id", "game_id", "channel",
            "sender_player_id", "recipient_player_id", "sent_ts",
            "message_text", "language", "sent_date"))

reports_raw_df = (
    with_uniforms(reports_full, ["dq1"], 902)
    .withColumn("report_category",
        F.when(c("dq1") < 0.003, F.lit("EXPLOITING")).otherwise(c("report_category")))  # DQ: invalid enum
    .select("report_id", "reporter_player_id", "reported_player_id",
            "report_date", "report_category", "evidence_message_id",
            "match_session_id", "game_id"))

# ─── Reference files (1 snapshot each) ────────────────────────────────────
def write_ref(df, path, fmt, **opts):
    """Write a small reference DF as a single file."""
    df.coalesce(1).write.mode("overwrite").format(fmt).options(**opts).save(f"{STAGE}/_ref")
    part = next(f for f in dbutils.fs.ls(f"{STAGE}/_ref") if f.name.startswith("part-"))
    dbutils.fs.cp(part.path, path)
    dbutils.fs.rm(f"{STAGE}/_ref", recurse=True)

write_ref(players_raw_df,  f"{RAW}/account/players/account_players_20261231.csv",  "csv", header="true")
write_ref(games_df,        f"{RAW}/game_svc/games/game_svc_games_20261231.json",   "json")
write_ref(events_df,       f"{RAW}/game_svc/live_ops_events/game_svc_live_ops_events_20261231.json", "json")
write_ref(devices_raw_df,  f"{RAW}/telemetry/devices/telemetry_devices_20261231.csv", "csv", header="true")
print("✓ 4 reference files written")

# ─── Daily files: stage with _date partition → bulk rename ─────────────────
def stage_and_rename(df, date_col, target_dir, prefix, fmt, n_days):
    """Partition by throwaway _date column (keeps date_col IN file data), then rename flat."""
    stage = f"{STAGE}/_daily_{prefix}"
    write_df = df.withColumn("_date", c(date_col)).repartitionByRange(n_days, "_date")
    opts = {"header": "true"} if fmt == "csv" else {}
    write_df.write.mode("overwrite").partitionBy("_date").format(fmt).options(**opts).save(stage)
    moves = []
    for pd in dbutils.fs.ls(stage):
        if not pd.name.startswith("_date="):
            continue
        ds = pd.name.replace("_date=", "").rstrip("/").replace("-", "")
        parts = [f for f in dbutils.fs.ls(pd.path) if f.name.startswith("part-")]
        for i, pf in enumerate(parts):
            sfx = f"_{i}" if i > 0 else ""
            moves.append((pf.path, f"{target_dir}/{prefix}_{ds}{sfx}.{fmt}"))
    def _mv(pair):
        dbutils.fs.mv(pair[0], pair[1])
    with ThreadPoolExecutor(max_workers=32) as pool:
        list(pool.map(_mv, moves))
    dbutils.fs.rm(stage, recurse=True)
    return len(moves)

n_days_cb  = chargebacks_raw_df.select("chargeback_date").distinct().count()
n_sessions = stage_and_rename(sessions_raw_df,     "session_date",    f"{RAW}/telemetry/sessions",    "telemetry_sessions",    "json", 731)
print(f"✓ sessions       → {n_sessions:,} files")
n_txns     = stage_and_rename(transactions_raw_df, "txn_date",        f"{RAW}/commerce/transactions", "commerce_transactions", "json", 731)
print(f"✓ transactions   → {n_txns:,} files")
n_cbs      = stage_and_rename(chargebacks_raw_df,  "chargeback_date", f"{RAW}/payments/chargebacks",  "payments_chargebacks",  "csv",  n_days_cb)
print(f"✓ chargebacks    → {n_cbs:,} files")
n_chat     = stage_and_rename(chat_raw_df,         "sent_date",       f"{RAW}/chat/chat_messages",    "chat_chat_messages",    "json", 731)
print(f"✓ chat           → {n_chat:,} files")
n_reports  = stage_and_rename(reports_raw_df,      "report_date",     f"{RAW}/safety/player_reports", "safety_player_reports", "json", 731)
print(f"✓ reports        → {n_reports:,} files")

# ─── Ground truth parquets ─────────────────────────────────────────────────
(players_full
    .select("player_id", "spend_tier", "is_fraud_ring", "ring_id",
            "is_toxic", "is_victim", "is_cheater", "has_bad_matchmaking",
            "activity_level", "churn_date", "is_churned")
    .write.mode("overwrite").parquet(f"{GT}/gt_players"))
(transactions_full
    .select("txn_id", "player_id", "game_id", "is_fraud_ring", "spend_tier")
    .join(chargebacks_full.select("txn_id", "fraud_type"), "txn_id", "left")
    .write.mode("overwrite").parquet(f"{GT}/gt_transactions"))
(chat_full
    .select("message_id", "category", "is_toxic", "is_banter",
            "severity", "is_obfuscated", "sender_is_toxic")
    .write.mode("overwrite").parquet(f"{GT}/gt_chat_labels"))
print("✓ 3 ground truth parquets written")

# ─── Cleanup intermediates ─────────────────────────────────────────────────
dbutils.fs.rm(WORK, recurse=True)
print("✓ _work cleaned up")

total = 4 + n_sessions + n_txns + n_cbs + n_chat + n_reports
print(f"\nTotal raw files: {total:,}  (plan target: ~3,659)")

In [0]:
print("=" * 70)
print("PHASE 1 VALIDATION — raw files + ground truth")
print("=" * 70)

sessions_v = spark.read.json(f"{RAW}/telemetry/sessions/")
txns_v     = spark.read.json(f"{RAW}/commerce/transactions/")
cbs_v      = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{RAW}/payments/chargebacks/")
chat_v     = spark.read.json(f"{RAW}/chat/chat_messages/")
devices_v  = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{RAW}/telemetry/devices/")
events_v   = spark.read.json(f"{RAW}/game_svc/live_ops_events/game_svc_live_ops_events_20261231.json")
gt_players = spark.read.parquet(f"{GT}/gt_players")
gt_txns    = spark.read.parquet(f"{GT}/gt_transactions")
gt_chat    = spark.read.parquet(f"{GT}/gt_chat_labels")
cb_flag    = cbs_v.select("txn_id", F.lit(True).alias("has_cb"))

# 1. Monthly MAU + revenue (last 6 months; expect upward trend)
print("\n--- 1. Monthly MAU and revenue (Jul-Dec 2026) ---")
mau = sessions_v.groupBy(F.substring("session_date", 1, 7).alias("month")).agg(F.countDistinct("player_id").alias("mau"))
rev = txns_v.groupBy(F.substring("txn_date", 1, 7).alias("month")).agg(F.round(F.sum("amount_usd"), 0).alias("revenue_usd"))
mau.join(rev, "month", "left").filter("month >= '2026-07'").orderBy("month").show()

# 2. Churn by victim status (expect ~2x higher for victims)
print("--- 2. Churn rate by is_victim ---")
gt_players.groupBy("is_victim").agg(
    F.count("*").alias("players"),
    F.round(F.avg(c("is_churned").cast("int")) * 100, 1).alias("churn_pct")
).orderBy("is_victim").show()

# 3. Revenue concentration (expect whales >> all tiers)
print("--- 3. Revenue concentration by spend tier ---")
txns_v.join(gt_txns.select("txn_id", "spend_tier"), "txn_id").groupBy("spend_tier").agg(
    F.count("*").alias("txns"),
    F.round(F.sum("amount_usd"), 0).alias("revenue_usd")
).orderBy("revenue_usd", ascending=False).show()

# 4. Cheater separation in shooters (expect ~2x accuracy + headshot)
print("--- 4. Cheater vs non-cheater: G01+G02 shooters ---")
(sessions_v.filter(c("game_id").isin("G01", "G02"))
 .join(gt_players.select("player_id", "is_cheater"), "player_id")
 .groupBy("is_cheater")
 .agg(F.round(F.avg("accuracy_pct"), 1).alias("avg_accuracy"),
      F.round(F.avg("headshot_pct"), 1).alias("avg_headshot"),
      F.count("*").alias("sessions"))
 .orderBy("is_cheater").show())

# 5. Chargeback lift: country mismatch
print("--- 5. Chargeback rate: billing vs IP country mismatch ---")
(txns_v.join(cb_flag, "txn_id", "left")
 .withColumn("has_cb", c("has_cb").isNotNull())
 .withColumn("country_mismatch", c("billing_country") != c("ip_country"))
 .groupBy("country_mismatch")
 .agg(F.count("*").alias("txns"),
      F.round(F.avg(c("has_cb").cast("int")) * 100, 2).alias("cb_rate_pct"))
 .orderBy("country_mismatch").show())

# 6. Toxicity rate by live-ops event type (expect esports/ranked_season highest)
print("--- 6. Toxic chat rate by event type ---")
evt = F.broadcast(events_v.select(c("event_id"), c("event_type")))
sess_evt = (sessions_v.select("session_id", "live_ops_event_id")
    .join(evt, sessions_v["live_ops_event_id"] == evt["event_id"], "left")
    .withColumn("etype", F.coalesce("event_type", F.lit("no_event"))))
(chat_v.join(gt_chat.select("message_id", "is_toxic"), "message_id")
 .join(sess_evt.select("session_id", "etype"), "session_id", "left")
 .groupBy("etype")
 .agg(F.count("*").alias("messages"),
      F.round(F.avg(c("is_toxic").cast("int")) * 100, 2).alias("toxic_pct"))
 .orderBy("toxic_pct", ascending=False).show())

# 7. Keyword-filter FP: banter misclassified as toxic
print("--- 7. Keyword-filter accuracy by message category ---")
KW = ["kill", "idiot", "trash", "loser", "noob", "quit", "stupid", "useless", "coward", "rekt"]
(chat_v.join(gt_chat.select("message_id", "category", "is_banter"), "message_id")
 .withColumn("kw_flag", F.expr(f"message_text RLIKE '(?i)({'|'.join(KW)})'" ))
 .groupBy("category")
 .agg(F.count("*").alias("messages"),
      F.sum(c("kw_flag").cast("int")).alias("kw_flagged"),
      F.round(F.sum(c("kw_flag").cast("int")) / F.count("*") * 100, 1).alias("flag_pct"))
 .orderBy("category").show())
print("=" * 70)
print("VALIDATION COMPLETE")

In [0]:
print("=" * 70)
print("INDEPENDENT VERIFICATION — re-read raw files via Spark")
print("=" * 70)

for entity, path, fmt in [
    ("sessions",     f"{RAW}/telemetry/sessions",             "json"),
    ("transactions", f"{RAW}/commerce/transactions",          "json"),
    ("chargebacks",  f"{RAW}/payments/chargebacks",           "csv_hdr"),
    ("chat",         f"{RAW}/chat/chat_messages",             "json"),
    ("reports",      f"{RAW}/safety/player_reports",          "json"),
]:
    if fmt == "csv_hdr":
        df = spark.read.option("header", "true").csv(path)
    else:
        df = spark.read.json(path)
    n_files = len(dbutils.fs.ls(path))
    n_rows  = df.count()
    print(f"{entity:16s}  files={n_files:4d}  rows={n_rows:>10,}")

# Reference + GT file counts
for label, path in [
    ("account/players",         f"{RAW}/account/players"),
    ("game_svc/games",          f"{RAW}/game_svc/games"),
    ("game_svc/live_ops_events",f"{RAW}/game_svc/live_ops_events"),
    ("telemetry/devices",       f"{RAW}/telemetry/devices"),
    ("GT gt_players",           f"{GT}/gt_players"),
    ("GT gt_transactions",      f"{GT}/gt_transactions"),
    ("GT gt_chat_labels",       f"{GT}/gt_chat_labels"),
]:
    items = dbutils.fs.ls(path)
    print(f"{label:28s}  items={len(items)}")
print("=" * 70)